<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/ALIGNN_FF_NVE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -q alignn

In [ ]:
#https://gist.github.com/leelasd/aaa517ac03d2f03bc1e181833e3a70fd
import numpy as np
from ase.lattice.cubic import FaceCenteredCubic
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.verlet import VelocityVerlet
from ase.calculators.dftb import Dftb
from ase.optimize import QuasiNewton,fire,gpmin,mdmin,LBFGS
from ase.io import write,read
from ase.build import molecule
from ase.optimize.minimahopping import MinimaHopping
from ase import *
from ase.optimize.basin import BasinHopping
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase import units
from ase import Atom, Atoms as AseAtoms
from alignn.ff.ff import AlignnAtomwiseCalculator,default_path,wt10_path,alignnff_fmult,fd_path,ForceField
model_path = wt10_path()
calc = AlignnAtomwiseCalculator(path=model_path)
# Set the momenta corresponding to T=300K
a = 3.597
atoms = AseAtoms([Atom('Cu', (0, 0, 0))],
              cell=0.5 * a * np.array([[1.0, 1.0, 0.0],
                                        [0.0, 1.0, 1.0],
                                        [1.0, 0.0, 1.0]]),
              pbc=True)

atoms.calc=calc
MaxwellBoltzmannDistribution(atoms, 300 * units.kB)

# We want to run MD with constant energy using the VelocityVerlet algorithm.
dyn = VelocityVerlet(atoms, 0.5 * units.fs)  # 5 fs time step.


def printenergy(a=atoms):  # store a reference to atoms in the definition.
    """Function to print the potential, kinetic and total energy."""
    epot = a.get_potential_energy() / len(a)
    ekin = a.get_kinetic_energy() / len(a)
    print('Energy per atom: Epot = %.3feV  Ekin = %.3feV (T=%3.0fK)  '
          'Etot = %.3feV' % (epot, ekin, ekin / (1.5 * units.kB), epot + ekin))

# Now run the dynamics
dyn.attach(printenergy, interval=10)
printenergy()
dyn.run(2000)
